### 1. Setup + Rolling Statistics per machine_id
**Tujuan:** Memuat df_sensor_labeled, menghitung rolling statistics (mean, std, max) untuk 6 sensor prioritas tinggi dengan window 24h dan 48h per machine_id.  
**Input:** data/interim/df_sensor_labeled.parquet, W_CRITICAL_HRS=24, W_WARNING_HRS=48  
**Output:** df (100.000×50)  +36 kolom rolling: 6 sensor × 2 window × 3 stats  
**Catatan:** Window 24h selaras W_CRITICAL_HRS, window 48h selaras W_WARNING_HRS  memastikan rolling features menangkap sinyal degradasi di kedua zona. Rolling std menghasilkan 20 NaN per sensor (baris pertama per mesin = undefined)  diselesaikan di Cell 2.  


In [1]:
# ============================================================
# FASE 4 - FEATURE ENGINEERING
# Cell 1: Setup + Rolling Statistics per machine_id
# ============================================================

import sys
import numpy as np
import pandas as pd
from pathlib import Path

# ----------------------------------------------------------
# [1] SYS.PATH SETUP
# Notebook berada di: notebooks/fase_4_feature_engineering/
# ML_ROOT = dua level ke atas
# ----------------------------------------------------------
NOTEBOOK_DIR = Path().resolve()
ML_ROOT      = NOTEBOOK_DIR.parent.parent
SRC_PATH     = ML_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# ----------------------------------------------------------
# [2] IMPORT CONFIG - single source of truth
# ----------------------------------------------------------
from config import (
    DATA_INTERIM_DIR,
    DATA_PROCESSED_DIR,
    GLOBAL_SEED,
    W_CRITICAL_HRS,
    W_WARNING_HRS,
)

np.random.seed(GLOBAL_SEED)

# ----------------------------------------------------------
# [3] LOAD DATA
# ----------------------------------------------------------
INPUT_PATH = DATA_INTERIM_DIR / "df_sensor_labeled.parquet"

df = pd.read_parquet(INPUT_PATH)

# Pastikan terurut per machine_id → timestamp (pra-syarat rolling)
df = (
    df
    .sort_values(["machine_id", "timestamp"], ascending=True)
    .reset_index(drop=True)
)

# ----------------------------------------------------------
# [4] KONFIRMASI SETUP
# ----------------------------------------------------------
SEP = "=" * 65
sep = "-" * 65

print(SEP)
print("  SETUP KONFIRMASI — FASE 4 FEATURE ENGINEERING")
print(SEP)
print(f"  SRC_PATH           : {SRC_PATH}")
print(f"  GLOBAL_SEED        : {GLOBAL_SEED}")
print(f"  W_CRITICAL_HRS     : {W_CRITICAL_HRS} jam")
print(f"  W_WARNING_HRS      : {W_WARNING_HRS} jam")
print(f"  INPUT_PATH         : {INPUT_PATH.name}")
print(f"  DATA_INTERIM_DIR   : {DATA_INTERIM_DIR}")
print(f"  DATA_PROCESSED_DIR : {DATA_PROCESSED_DIR}")
print(f"\n  Shape df           : {df.shape}")
print(f"  Kolom df           : {list(df.columns)}")
print(f"\n  dtype timestamp    : {df['timestamp'].dtype}")
print(f"  timestamp min      : {df['timestamp'].min()}")
print(f"  timestamp max      : {df['timestamp'].max()}")
print(SEP)

# ============================================================
# ROLLING STATISTICS PER MACHINE_ID
# ============================================================

# ----------------------------------------------------------
# PARAMETER — selaras dengan W_CRITICAL dan W_WARNING di config
# ----------------------------------------------------------
HIGH_PRIORITY_SENSORS = [
    "temperature",
    "vibration",
    "pressure",
    "rpm",
    "power_consumption",
    "noise_level",
]
WINDOW_SIZES = [24, 48]   # jam — selaras W_CRITICAL (24h) & W_WARNING (48h)

# Hanya gunakan sensor yang benar-benar ada di df (defensive)
sensors_available = [s for s in HIGH_PRIORITY_SENSORS if s in df.columns]

# ----------------------------------------------------------
# CATAT SHAPE SEBELUM
# ----------------------------------------------------------
shape_before = df.shape

# ----------------------------------------------------------
# ROLLING STATS — groupby machine_id, min_periods=1
# ----------------------------------------------------------
# Kumpulkan nama kolom baru yang akan dibentuk
rolling_cols_expected = []

for sensor in sensors_available:
    for window in WINDOW_SIZES:
        rolling_cols_expected.append(f"{sensor}_roll_mean_{window}h")
        rolling_cols_expected.append(f"{sensor}_roll_std_{window}h")
        rolling_cols_expected.append(f"{sensor}_roll_max_{window}h")

# Komputasi rolling per machine_id
for sensor in sensors_available:
    for window in WINDOW_SIZES:
        # Groupby machine_id, hitung rolling dalam satu .agg() per stat
        rolled = (
            df.groupby("machine_id", group_keys=False)[sensor]
            .rolling(window=window, min_periods=1)
            .agg(["mean", "std", "max"])
            .reset_index(drop=True)
        )

        df[f"{sensor}_roll_mean_{window}h"] = rolled["mean"].values
        df[f"{sensor}_roll_std_{window}h"]  = rolled["std"].values
        df[f"{sensor}_roll_max_{window}h"]  = rolled["max"].values

# ----------------------------------------------------------
# VALIDASI HASIL ROLLING
# ----------------------------------------------------------
# Kolom rolling yang berhasil terbentuk
rolling_cols_formed = [c for c in rolling_cols_expected if c in df.columns]

shape_after = df.shape

print(f"\n{SEP}")
print("  VALIDASI ROLLING STATISTICS")
print(SEP)

print(f"\n{sep}")
print("  [A] RINGKASAN KOLOM ROLLING")
print(sep)
print(f"\n  Sensor diproses         : {sensors_available}")
print(f"  Window sizes            : {WINDOW_SIZES} jam")
print(f"  Stats per (sensor×window): mean, std, max")
print(f"  Jumlah kolom baru       : {len(rolling_cols_formed)}")
print(f"    (Expected : {len(rolling_cols_expected)})")

print(f"\n{sep}")
print("  [B] DAFTAR KOLOM ROLLING BARU")
print(sep)
print()
for col in rolling_cols_formed:
    print(f"    {col}")

print(f"\n{sep}")
print("  [C] CEK MISSING VALUES PADA KOLOM ROLLING")
print(sep)
print()

nan_counts = df[rolling_cols_formed].isna().sum()
nan_nonzero = nan_counts[nan_counts > 0]

if nan_nonzero.empty:
    print("  [OK] Tidak ada NaN pada seluruh kolom rolling (min_periods=1 bekerja).")
else:
    print(f"  [WARN] Ditemukan NaN pada {len(nan_nonzero)} kolom:")
    for col, cnt in nan_nonzero.items():
        print(f"    {col:<45} : {cnt:,} NaN")

print(f"\n{sep}")
print("  [D] CEK SHAPE (harus tetap 100,000 baris)")
print(sep)
print(f"\n  Shape SEBELUM rolling  : {shape_before}")
print(f"  Shape SESUDAH rolling  : {shape_after}")

row_check = "[OK]" if shape_before[0] == shape_after[0] else "[WARN]"
print(f"  {row_check} Jumlah baris tidak berubah : {shape_before[0] == shape_after[0]}")

print(f"\n{SEP}")
print("  [OK] Rolling Statistics selesai. df siap untuk Cell 2.")
print(SEP)


  SETUP KONFIRMASI — FASE 4 FEATURE ENGINEERING
  SRC_PATH           : C:\PORTFOLIO\PROJECTS\PBL\Predictive Maintenance\projects\predictive-maintenance-monorepo\machine_learning\src
  GLOBAL_SEED        : 42
  W_CRITICAL_HRS     : 24 jam
  W_WARNING_HRS      : 48 jam
  INPUT_PATH         : df_sensor_labeled.parquet
  DATA_INTERIM_DIR   : C:\PORTFOLIO\PROJECTS\PBL\Predictive Maintenance\projects\predictive-maintenance-monorepo\machine_learning\data\interim
  DATA_PROCESSED_DIR : C:\PORTFOLIO\PROJECTS\PBL\Predictive Maintenance\projects\predictive-maintenance-monorepo\machine_learning\data\processed

  Shape df           : (100000, 14)
  Kolom df           : ['timestamp', 'machine_id', 'temperature', 'vibration', 'pressure', 'rpm', 'power_consumption', 'noise_level', 'humidity', 'operating_hours', 'failure', 'health_label', 'health_label_confirmed', 'health_label_encoded']

  dtype timestamp    : datetime64[ns]
  timestamp min      : 2025-07-01 00:00:00
  timestamp max      : 2026-01-25 

### 2. Fix NaN pada Kolom Rolling STD
**Tujuan:** Mengisi NaN pada 12 kolom rolling std (baris pertama tiap mesin) dengan 0 menggunakan fillna(0).  
**Input:** df (100.000×50) dari Cell 1  
**Output:** df (100.000×50)  0 NaN di seluruh DataFrame  
**Catatan:** std dari 1 sampel = mathematically undefined  NaN pada baris pertama per mesin (20 baris × 12 kolom = 240 NaN). Fill dengan 0 = 'tidak ada variabilitas'  semantis tepat untuk awal pengukuran.  


In [2]:
# FASE 4 - Cell 2: Fix NaN pada kolom Rolling STD
# rolling().std() menghasilkan NaN pada baris pertama per machine_id
# karena std dari 1 sampel = undefined. Fix: fillna(0)

SEP = "=" * 65
sep = "-" * 65

# ----------------------------------------------------------
# [1] IDENTIFIKASI KOLOM ROLLING STD
# ----------------------------------------------------------
std_cols = [col for col in df.columns if "_roll_std_" in col]

print(SEP)
print("  FIX NaN — KOLOM ROLLING STD")
print(SEP)
print(f"\n  Jumlah kolom _roll_std_ ditemukan : {len(std_cols)}")
for col in std_cols:
    print(f"    {col}")

# ----------------------------------------------------------
# [A] LAPORAN NaN SEBELUM FIX
# ----------------------------------------------------------
print(f"\n{sep}")
print("  [A] NaN SEBELUM FIX")
print(sep)
print()

nan_before         = df[std_cols].isna().sum()
nan_before_nonzero = nan_before[nan_before > 0]

if nan_before_nonzero.empty:
    print("  Tidak ada NaN pada kolom _roll_std_ (tidak perlu fix).")
else:
    print(f"  Kolom dengan NaN ({len(nan_before_nonzero)} kolom):")
    for col, cnt in nan_before_nonzero.items():
        print(f"    {col:<45} : {cnt:,} NaN")

# ----------------------------------------------------------
# [3] FIX: FILLNA(0) PADA SEMUA KOLOM _roll_std_
# ----------------------------------------------------------
df[std_cols] = df[std_cols].fillna(0)

# ----------------------------------------------------------
# [B] VERIFIKASI SETELAH FIX
# ----------------------------------------------------------
print(f"\n{sep}")
print("  [B] VERIFIKASI SETELAH FIX")
print(sep)
print()

total_nan_df = int(df.isna().sum().sum())

if total_nan_df == 0:
    print("  [OK] Total NaN di seluruh df    : 0")
    print("  [OK] Seluruh kolom rolling STD berhasil di-fill dengan 0.")
else:
    print(f"  [WARN] Masih ada {total_nan_df:,} NaN di df. Detail per kolom:")
    remaining_nan = df.isna().sum()
    remaining_nan = remaining_nan[remaining_nan > 0]
    for col, cnt in remaining_nan.items():
        print(f"    {col:<45} : {cnt:,} NaN")

# ----------------------------------------------------------
# [C] CEK SHAPE FINAL
# ----------------------------------------------------------
print(f"\n{sep}")
print("  [C] CEK SHAPE FINAL")
print(sep)
print(f"\n  Shape df : {df.shape}")

shape_ok   = df.shape == (100_000, 50)
shape_flag = "[OK]" if shape_ok else "[WARN]"
print(f"  {shape_flag} Shape sesuai target (100,000 × 50) : {shape_ok}")

print(f"\n{SEP}")
print("  [OK] Fix NaN rolling STD selesai. df siap untuk Cell 3.")
print(SEP)


  FIX NaN — KOLOM ROLLING STD

  Jumlah kolom _roll_std_ ditemukan : 12
    temperature_roll_std_24h
    temperature_roll_std_48h
    vibration_roll_std_24h
    vibration_roll_std_48h
    pressure_roll_std_24h
    pressure_roll_std_48h
    rpm_roll_std_24h
    rpm_roll_std_48h
    power_consumption_roll_std_24h
    power_consumption_roll_std_48h
    noise_level_roll_std_24h
    noise_level_roll_std_48h

-----------------------------------------------------------------
  [A] NaN SEBELUM FIX
-----------------------------------------------------------------

  Kolom dengan NaN (12 kolom):
    temperature_roll_std_24h                      : 20 NaN
    temperature_roll_std_48h                      : 20 NaN
    vibration_roll_std_24h                        : 20 NaN
    vibration_roll_std_48h                        : 20 NaN
    pressure_roll_std_24h                         : 20 NaN
    pressure_roll_std_48h                         : 20 NaN
    rpm_roll_std_24h                              : 2

### 3. Lag Features per machine_id
**Tujuan:** Menghitung lag features (shift 6h, 12h, 24h) untuk 6 sensor prioritas tinggi per machine_id, lalu fill NaN dengan ffill + bfill.  
**Input:** df (100.000×50) dari Cell 2  
**Output:** df (100.000×68)  +18 kolom lag: 6 sensor × 3 lag size  
**Catatan:** Lag features memberi model konteks temporal historis: 'nilai sensor 6/12/24 jam lalu'. Fill strategy: ffill (gunakan nilai terdekat sebelumnya)  bfill (untuk NaN di baris awal mesin).  


In [3]:
# ============================================================
# FASE 4 - FEATURE ENGINEERING
# Cell 3: Lag Features per machine_id
# ============================================================
# Lag features menangkap nilai sensor di t-n jam sebelumnya,
# memberi model konteks temporal historis yang kaya.
# Fill strategy: ffill dulu (gunakan nilai terdekat sebelumnya),
# lalu bfill untuk sisa NaN di awal mesin (awal dataset).
# ============================================================

SEP = "=" * 65
sep = "-" * 65

# ----------------------------------------------------------
# PARAMETER
# ----------------------------------------------------------
HIGH_PRIORITY_SENSORS = [
    "temperature",
    "vibration",
    "pressure",
    "rpm",
    "power_consumption",
    "noise_level",
]
LAG_SIZES = [6, 12, 24]  # jam

# Defensive: hanya proses sensor yang ada di df
sensors_available = [s for s in HIGH_PRIORITY_SENSORS if s in df.columns]

# Kumpulkan nama kolom lag yang diharapkan
lag_cols_expected = []
for sensor in sensors_available:
    for lag in LAG_SIZES:
        lag_cols_expected.append(f"{sensor}_lag_{lag}h")

# ----------------------------------------------------------
# CATAT SHAPE SEBELUM
# ----------------------------------------------------------
shape_before = df.shape

print(SEP)
print("  LAG FEATURES — FASE 4 FEATURE ENGINEERING")
print(SEP)
print(f"\\n  Sensor diproses  : {sensors_available}")
print(f"  Lag sizes        : {LAG_SIZES} jam")
print(f"  Kolom lag target : {len(lag_cols_expected)}")
print(f"  Shape sebelum    : {shape_before}")

# ----------------------------------------------------------
# KOMPUTASI LAG PER MACHINE_ID
# ----------------------------------------------------------
for sensor in sensors_available:
    for lag in LAG_SIZES:
        col_name = f"{sensor}_lag_{lag}h"

        # shift(lag) per machine_id → NaN di baris awal tiap mesin
        df[col_name] = (
            df.groupby("machine_id", group_keys=False)[sensor]
            .shift(lag)
        )

# ----------------------------------------------------------
# FILL NaN HASIL LAG — ffill → bfill per machine_id
# ----------------------------------------------------------
lag_cols_formed = [c for c in lag_cols_expected if c in df.columns]

for col in lag_cols_formed:
    df[col] = (
        df.groupby("machine_id", group_keys=False)[col]
        .transform(lambda s: s.ffill().bfill())
    )

# ----------------------------------------------------------
# VALIDASI
# ----------------------------------------------------------
shape_after = df.shape

print(f"\\n{SEP}")
print("  VALIDASI LAG FEATURES")
print(SEP)

# [A] Ringkasan kolom
print(f"\\n{sep}")
print("  [A] RINGKASAN KOLOM LAG")
print(sep)
print(f"\\n  Kolom lag terbentuk : {len(lag_cols_formed)}")
print(f"  (Expected           : {len(lag_cols_expected)})")

# [B] Daftar kolom lag
print(f"\\n{sep}")
print("  [B] DAFTAR KOLOM LAG BARU")
print(sep)
print()
for col in lag_cols_formed:
    print(f"    {col}")

# [C] Cek NaN setelah fill
print(f"\\n{sep}")
print("  [C] CEK NaN SETELAH FFILL + BFILL")
print(sep)
print()

nan_lag    = df[lag_cols_formed].isna().sum()
nan_nonzero = nan_lag[nan_lag > 0]

if nan_nonzero.empty:
    print("  [OK] Total NaN pada kolom lag : 0")
    print("  [OK] ffill + bfill berhasil mengisi semua NaN.")
else:
    print(f"  [WARN] Masih ada NaN pada {len(nan_nonzero)} kolom lag:")
    for col, cnt in nan_nonzero.items():
        print(f"    {col:<40} : {cnt:,} NaN")

# [D] Shape final
print(f"\\n{sep}")
print("  [D] CEK SHAPE FINAL")
print(sep)
print(f"\\n  Shape SEBELUM : {shape_before}")
print(f"  Shape SESUDAH : {shape_after}")

shape_ok   = shape_after == (100_000, 68)
shape_flag = "[OK]" if shape_ok else "[WARN]"
print(f"  {shape_flag} Shape sesuai target (100,000 × 68) : {shape_ok}")

print(f"\\n{SEP}")
print("  [OK] Lag Features selesai. df siap untuk Cell 4.")
print(SEP)

  LAG FEATURES — FASE 4 FEATURE ENGINEERING
\n  Sensor diproses  : ['temperature', 'vibration', 'pressure', 'rpm', 'power_consumption', 'noise_level']
  Lag sizes        : [6, 12, 24] jam
  Kolom lag target : 18
  Shape sebelum    : (100000, 50)
\n=================================================================
  VALIDASI LAG FEATURES
\n-----------------------------------------------------------------
  [A] RINGKASAN KOLOM LAG
-----------------------------------------------------------------
\n  Kolom lag terbentuk : 18
  (Expected           : 18)
\n-----------------------------------------------------------------
  [B] DAFTAR KOLOM LAG BARU
-----------------------------------------------------------------

    temperature_lag_6h
    temperature_lag_12h
    temperature_lag_24h
    vibration_lag_6h
    vibration_lag_12h
    vibration_lag_24h
    pressure_lag_6h
    pressure_lag_12h
    pressure_lag_24h
    rpm_lag_6h
    rpm_lag_12h
    rpm_lag_24h
    power_consumption_lag_6h
    powe

4.3 Cross-Sensor Ratios & Degradation Proxy

### 4. Cross-Sensor Ratios & Degradation Proxy
**Tujuan:** Menghitung 4 rasio cross-sensor (temp/vibration, power/rpm, pressure/temp, noise/vibration) dan 1 fitur degradasi (hours_since_last_maint) dari maintenance logs.  
**Input:** df (100.000×68) dari Cell 3, data/raw/maintenance_logs.csv  
**Output:** df (100.000×73)  +4 kolom ratio + 1 kolom degradation proxy  
**Catatan:** Rasio cross-sensor menangkap interaksi antar sensor yang tidak bisa ditangkap individual. hours_since_last_maint: baris dengan nilai -1 = 2.856 (belum ada maintenance tercatat sebelumnya)  acceptable, bukan bug.  


In [4]:
# ============================================================
# FASE 4 — Cell 4: Cross-Sensor Ratios & Degradation Proxy
# ============================================================

import numpy as np
import pandas as pd
import sys
from pathlib import Path

# Pastikan SRC_PATH tersedia (cell dapat dijalankan mandiri)
NOTEBOOK_DIR = Path().resolve()
ML_ROOT      = NOTEBOOK_DIR.parent.parent
SRC_PATH     = ML_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from config import MAINTENANCE_FILE

SEP = "=" * 65
sep = "-" * 65

# ════════════════════════════════════════════════════════════
# BAGIAN 1 — CROSS-SENSOR RATIOS
# ════════════════════════════════════════════════════════════

print(SEP)
print("  BAGIAN 1 — CROSS-SENSOR RATIOS")
print(SEP)

shape_before = df.shape

# --- Buat 4 kolom ratio ---
df["temp_per_vibration"]  = df["temperature"]       / (df["vibration"]        + 1e-9)
df["power_per_rpm"]       = df["power_consumption"] / (df["rpm"]              + 1e-9)
df["pressure_per_temp"]   = df["pressure"]          / (df["temperature"]      + 1e-9)
df["noise_per_vibration"] = df["noise_level"]       / (df["vibration"]        + 1e-9)

ratio_cols = [
    "temp_per_vibration",
    "power_per_rpm",
    "pressure_per_temp",
    "noise_per_vibration",
]

# --- Cek & tangani NaN dan Inf ---
print(f"\n{sep}")
print("  [A] CEK & HANDLE NaN / Inf PADA KOLOM RATIO")
print(sep)
print()

for col in ratio_cols:
    n_inf = int(np.isinf(df[col]).sum())
    n_nan = int(df[col].isna().sum())

    # Replace Inf → NaN → fill dengan median
    if n_inf > 0:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)

    col_median = df[col].median()

    if df[col].isna().any():
        df[col] = df[col].fillna(col_median)

    n_nan_after = int(df[col].isna().sum())
    print(f"  {col:<28} | Inf={n_inf:>4}  NaN_before={n_nan:>4}  "
          f"filled_with_median={col_median:.4f}  NaN_after={n_nan_after}")

# --- Statistik singkat per kolom ratio ---
print(f"\n{sep}")
print("  [B] STATISTIK KOLOM RATIO (min / max / mean)")
print(sep)
print()
for col in ratio_cols:
    mn  = df[col].min()
    mx  = df[col].max()
    avg = df[col].mean()
    print(f"  {col:<28} | min={mn:>12.4f}  max={mx:>12.4f}  mean={avg:>12.4f}")

# ════════════════════════════════════════════════════════════
# BAGIAN 2 — DEGRADATION PROXY (hours_since_last_maint)
# ════════════════════════════════════════════════════════════

print(f"\n{SEP}")
print("  BAGIAN 2 — DEGRADATION PROXY")
print(SEP)

# --- Load maintenance log ---
df_maintenance = pd.read_csv(
    MAINTENANCE_FILE,
    parse_dates=["date"],
    dtype={"machine_id": str},
)

print(f"\n  MAINTENANCE_FILE  : {MAINTENANCE_FILE.name}")
print(f"  Shape df_maint    : {df_maintenance.shape}")
print(f"  Kolom             : {list(df_maintenance.columns)}")

# --- Siapkan df_maint_clean ---
df_maint_clean = (
    df_maintenance[["date", "machine_id"]]
    .sort_values("date", ascending=True)
    .reset_index(drop=True)
)

# --- Siapkan df_left (timestamp sudah terurut dari Cell 1) ---
df_left = df[["timestamp", "machine_id"]].copy()
df_left = df_left.sort_values("timestamp", ascending=True)

# --- merge_asof: cari maintenance terakhir sebelum setiap timestamp ---
df_merged = pd.merge_asof(
    left      = df_left,
    right     = df_maint_clean,
    left_on   = "timestamp",
    right_on  = "date",
    by        = "machine_id",
    direction = "backward",
)

# --- Hitung hours_since_last_maint ---
df["hours_since_last_maint"] = (
    (df_merged["timestamp"] - df_merged["date"])
    .dt.total_seconds() / 3600
).values

# Fill NaN dengan -1 (belum pernah ada maintenance sebelumnya)
df["hours_since_last_maint"] = df["hours_since_last_maint"].fillna(-1)

# --- Validasi Bagian 2 ---
print(f"\n{sep}")
print("  [C] STATISTIK hours_since_last_maint")
print(sep)

col_deg = df["hours_since_last_maint"]
print(f"\n  min    : {col_deg.min():>12.2f} jam")
print(f"  max    : {col_deg.max():>12.2f} jam")
print(f"  mean   : {col_deg.mean():>12.2f} jam")
print(f"  median : {col_deg.median():>12.2f} jam")

n_minus1 = int((col_deg == -1).sum())
print(f"\n  Baris dengan nilai -1 (belum ada maint) : {n_minus1:,}")

# --- Shape final ---
shape_after = df.shape

print(f"\n{sep}")
print("  [D] CEK SHAPE FINAL")
print(sep)
print(f"\n  Shape SEBELUM : {shape_before}")
print(f"  Shape SESUDAH : {shape_after}")

shape_ok   = shape_after == (100_000, 73)
shape_flag = "[OK]" if shape_ok else "[WARN]"
print(f"  {shape_flag} Shape sesuai target (100,000 × 73) : {shape_ok}")

print(f"\n{SEP}")
print("  [OK] Cross-Sensor Ratios & Degradation Proxy selesai.")
print("  [OK] df siap untuk Cell 5 (Export).")
print(SEP)


  BAGIAN 1 — CROSS-SENSOR RATIOS

-----------------------------------------------------------------
  [A] CEK & HANDLE NaN / Inf PADA KOLOM RATIO
-----------------------------------------------------------------

  temp_per_vibration           | Inf=   0  NaN_before=   0  filled_with_median=158.2979  NaN_after=0
  power_per_rpm                | Inf=   0  NaN_before=   0  filled_with_median=0.0313  NaN_after=0
  pressure_per_temp            | Inf=   0  NaN_before=   0  filled_with_median=1.3992  NaN_after=0
  noise_per_vibration          | Inf=   0  NaN_before=   0  filled_with_median=153.8529  NaN_after=0

-----------------------------------------------------------------
  [B] STATISTIK KOLOM RATIO (min / max / mean)
-----------------------------------------------------------------

  temp_per_vibration           | min=  -2690.0001  max=   7129.9993  mean=    166.0363
  power_per_rpm                | min=      0.0208  max=      0.0492  mean=      0.0315
  pressure_per_temp            |

4.4 NLP: Damage Category & Severity Score

### 4. Cross-Sensor Ratios & Degradation Proxy
**Tujuan:** Menghitung 4 rasio cross-sensor (temp/vibration, power/rpm, pressure/temp, noise/vibration) dan 1 fitur degradasi (hours_since_last_maint) dari maintenance logs.  
**Input:** df (100.000×68) dari Cell 3, data/raw/maintenance_logs.csv  
**Output:** df (100.000×73)  +4 kolom ratio + 1 kolom degradation proxy  
**Catatan:** Rasio cross-sensor menangkap interaksi antar sensor yang tidak bisa ditangkap individual. hours_since_last_maint: baris dengan nilai -1 = 2.856 (belum ada maintenance tercatat sebelumnya)  acceptable, bukan bug.  


In [5]:
# FASE 4 — Cell 5: NLP Text Mining (damage_category & severity_score)
# Kolom di maintenance_logs.csv yang digunakan:
#   technician_notes, maintenance_type, downtime_hours

import numpy as np
import pandas as pd
import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
ML_ROOT      = NOTEBOOK_DIR.parent.parent
SRC_PATH     = ML_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from config import MAINTENANCE_FILE

SEP = "=" * 65
sep = "-" * 65

# ════════════════════════════════════════════════════════════
# BAGIAN 1 — EKSTRAKSI FITUR DARI technician_notes
# ════════════════════════════════════════════════════════════

print(SEP)
print("  BAGIAN 1 — NLP TEXT MINING: MAINTENANCE LOGS")
print(SEP)

df_maintenance = pd.read_csv(
    MAINTENANCE_FILE,
    parse_dates=["date"],
    dtype={"machine_id": str},
)

print(f"\n  MAINTENANCE_FILE : {MAINTENANCE_FILE.name}")
print(f"  Shape df_maint   : {df_maintenance.shape}")

# ----------------------------------------------------------
# LANGKAH 1A — damage_category (keyword matching)
# ----------------------------------------------------------
CATEGORY_KEYWORDS = {
    "Mechanical"  : ["belt", "bearing", "pulley", "poros",
                     "gear", "kopling", "rotor", "impeller",
                     "putus", "patah", "retak", "aus"],
    "Electrical"  : ["listrik", "sensor", "kabel", "panel",
                     "motor", "kontaktor", "relay", "sekering",
                     "korsleting", "tegangan"],
    "Thermal"     : ["panas", "overheat", "suhu", "temperatur",
                     "cooling", "pendingin", "radiator"],
    "Lubrication" : ["oli", "pelumas", "grease", "gemuk",
                     "kering", "gesekan"],
    "Routine"     : ["inspeksi", "rutin", "cek", "periksa",
                     "ganti oli", "servis"],
}

def assign_damage_category(notes: str) -> str:
    """Assign kategori kerusakan berdasarkan keyword matching."""
    if not isinstance(notes, str):
        return "Unknown"
    text = notes.lower()
    for category, keywords in CATEGORY_KEYWORDS.items():
        if any(kw in text for kw in keywords):
            return category
    return "Unknown"

df_maintenance["damage_category"] = (
    df_maintenance["technician_notes"].apply(assign_damage_category)
)

# ----------------------------------------------------------
# LANGKAH 1B — severity_score (1-3)
# ----------------------------------------------------------
def assign_severity_score(row) -> int:
    """
    Score 1 (Ringan) : Preventive OR downtime_hours < 4
    Score 2 (Sedang) : Corrective AND 4 <= downtime_hours <= 8
    Score 3 (Berat)  : Corrective AND downtime_hours > 8
    """
    m_type   = str(row["maintenance_type"]).strip()
    downtime = float(row["downtime_hours"])

    if m_type == "Preventive" or downtime < 4:
        return 1
    elif m_type == "Corrective" and 4 <= downtime <= 8:
        return 2
    else:
        return 3

df_maintenance["severity_score"] = df_maintenance.apply(
    assign_severity_score, axis=1
)

# ----------------------------------------------------------
# VALIDASI BAGIAN 1
# ----------------------------------------------------------
print(f"\n{sep}")
print("  [A] DISTRIBUSI damage_category")
print(sep)
print()

cat_counts = df_maintenance["damage_category"].value_counts()
cat_pct    = (df_maintenance["damage_category"].value_counts(normalize=True) * 100).round(2)
for cat in cat_counts.index:
    print(f"    {cat:<14} : {cat_counts[cat]:>4}  ({cat_pct[cat]:.2f}%)")

print(f"\n{sep}")
print("  [B] DISTRIBUSI severity_score")
print(sep)
print()

sev_counts = df_maintenance["severity_score"].value_counts().sort_index()
sev_pct    = (df_maintenance["severity_score"].value_counts(normalize=True) * 100).sort_index().round(2)
label_map  = {1: "Ringan", 2: "Sedang", 3: "Berat"}
for score in sev_counts.index:
    print(f"    Score {score} ({label_map.get(score,'?'):<6}) : "
          f"{sev_counts[score]:>4}  ({sev_pct[score]:.2f}%)")

print(f"\n{sep}")
print("  [C] 5 CONTOH BARIS HASIL EKSTRAKSI")
print(sep)
preview_cols = ["machine_id", "technician_notes", "damage_category", "severity_score"]
print()
print(df_maintenance[preview_cols].head(5).to_string(index=False))

# ════════════════════════════════════════════════════════════
# BAGIAN 2 — TEMPORAL MERGE KE df SENSOR
# ════════════════════════════════════════════════════════════

print(f"\n{SEP}")
print("  BAGIAN 2 — TEMPORAL MERGE KE df SENSOR")
print(SEP)

shape_before = df.shape

# Siapkan df_maint_nlp
df_maint_nlp = (
    df_maintenance[["date", "machine_id", "damage_category", "severity_score"]]
    .sort_values("date", ascending=True)
    .reset_index(drop=True)
)

# Siapkan df_left (timestamp sudah terurut dari Cell 1)
df_left = (
    df[["timestamp", "machine_id"]]
    .copy()
    .sort_values("timestamp", ascending=True)
)

# merge_asof: cari maintenance terakhir sebelum setiap timestamp
merged = pd.merge_asof(
    left      = df_left,
    right     = df_maint_nlp,
    left_on   = "timestamp",
    right_on  = "date",
    by        = "machine_id",
    direction = "backward",
)

# Assign ke df
df["damage_category"] = merged["damage_category"].values
df["severity_score"]  = merged["severity_score"].values

# Handle NaN (sebelum maintenance pertama per mesin)
df["damage_category"] = df["damage_category"].fillna("Unknown")
df["severity_score"]  = df["severity_score"].fillna(0)

# ----------------------------------------------------------
# VALIDASI BAGIAN 2
# ----------------------------------------------------------
print(f"\n{sep}")
print("  [D] DISTRIBUSI damage_category DI df SENSOR")
print(sep)
print()
cat2 = df["damage_category"].value_counts()
for cat in cat2.index:
    pct = cat2[cat] / len(df) * 100
    print(f"    {cat:<14} : {cat2[cat]:>7,}  ({pct:.2f}%)")

print(f"\n{sep}")
print("  [E] DISTRIBUSI severity_score DI df SENSOR")
print(sep)
print()
sev2 = df["severity_score"].value_counts().sort_index()
score_label = {0: "Belum maint", 1: "Ringan", 2: "Sedang", 3: "Berat"}
for score in sev2.index:
    pct = sev2[score] / len(df) * 100
    print(f"    Score {score} ({score_label.get(score,'?'):<11}) : "
          f"{sev2[score]:>7,}  ({pct:.2f}%)")

print(f"\n{sep}")
print("  [F] CEK NaN SETELAH FILL")
print(sep)
nan_cat = int(df["damage_category"].isna().sum())
nan_sev = int(df["severity_score"].isna().sum())
print(f"\n  NaN damage_category : {nan_cat}")
print(f"  NaN severity_score  : {nan_sev}")
flag = "[OK]" if (nan_cat + nan_sev) == 0 else "[WARN]"
print(f"  {flag} Total NaN kedua kolom : {nan_cat + nan_sev}")

print(f"\n{sep}")
print("  [G] CEK SHAPE FINAL")
print(sep)
shape_after = df.shape
print(f"\n  Shape SEBELUM : {shape_before}")
print(f"  Shape SESUDAH : {shape_after}")
shape_ok   = shape_after == (100_000, 75)
shape_flag = "[OK]" if shape_ok else "[WARN]"
print(f"  {shape_flag} Shape sesuai target (100,000 × 75) : {shape_ok}")

print(f"\n{SEP}")
print("  [OK] NLP Text Mining selesai. df siap untuk Cell 6 (Export).")
print(SEP)


  BAGIAN 1 — NLP TEXT MINING: MAINTENANCE LOGS

  MAINTENANCE_FILE : maintenance_logs.csv
  Shape df_maint   : (500, 8)

-----------------------------------------------------------------
  [A] DISTRIBUSI damage_category
-----------------------------------------------------------------

    Mechanical     :  163  (32.60%)
    Electrical     :  154  (30.80%)
    Lubrication    :   62  (12.40%)
    Unknown        :   52  (10.40%)
    Routine        :   51  (10.20%)
    Thermal        :   18  (3.60%)

-----------------------------------------------------------------
  [B] DISTRIBUSI severity_score
-----------------------------------------------------------------

    Score 1 (Ringan) :  255  (51.00%)
    Score 3 (Berat ) :  245  (49.00%)

-----------------------------------------------------------------
  [C] 5 CONTOH BARIS HASIL EKSTRAKSI
-----------------------------------------------------------------

machine_id                              technician_notes damage_category  severity_sc

### 4. Cross-Sensor Ratios & Degradation Proxy
**Tujuan:** Menghitung 4 rasio cross-sensor (temp/vibration, power/rpm, pressure/temp, noise/vibration) dan 1 fitur degradasi (hours_since_last_maint) dari maintenance logs.  
**Input:** df (100.000×68) dari Cell 3, data/raw/maintenance_logs.csv  
**Output:** df (100.000×73)  +4 kolom ratio + 1 kolom degradation proxy  
**Catatan:** Rasio cross-sensor menangkap interaksi antar sensor yang tidak bisa ditangkap individual. hours_since_last_maint: baris dengan nilai -1 = 2.856 (belum ada maintenance tercatat sebelumnya)  acceptable, bukan bug.  


In [7]:
# ============================================================
# FASE 4 — Cell Diagnostik: Investigasi Distribusi Downtime
# Tujuan: Verifikasi mengapa severity_score = 2 tidak muncul
# Ref defect log: DFT-03
# ============================================================

import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
ML_ROOT      = NOTEBOOK_DIR.parent.parent
SRC_PATH     = ML_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import pandas as pd
from config import MAINTENANCE_FILE

SEP = "=" * 65
sep = "-" * 65

# ----------------------------------------------------------
# LOAD
# ----------------------------------------------------------
df_maintenance = pd.read_csv(
    MAINTENANCE_FILE,
    parse_dates=["date"],
    dtype={"machine_id": str},
)

# ----------------------------------------------------------
# [1] FILTER CORRECTIVE ONLY
# ----------------------------------------------------------
df_corrective = df_maintenance[
    df_maintenance["maintenance_type"] == "Corrective"
].copy()

print(SEP)
print("  INVESTIGASI DISTRIBUSI DOWNTIME — CORRECTIVE ONLY")
print(SEP)
print(f"\n  Total baris Corrective : {len(df_corrective):,}")
print(f"  Total baris semua tipe : {len(df_maintenance):,}")

# ----------------------------------------------------------
# [2] STATISTIK downtime_hours
# ----------------------------------------------------------
print(f"\n{sep}")
print("  [A] STATISTIK downtime_hours (Corrective)")
print(sep)
dh = df_corrective["downtime_hours"]
print(f"\n  min    : {dh.min():.2f} jam")
print(f"  max    : {dh.max():.2f} jam")
print(f"  mean   : {dh.mean():.2f} jam")
print(f"  median : {dh.median():.2f} jam")
print(f"\n  value_counts (top 20):")
print()
vc = dh.value_counts().sort_index()
for val, cnt in vc.items():
    print(f"    {val:>6.1f} jam  →  {cnt:>3} baris")

# ----------------------------------------------------------
# [3] BARIS DENGAN downtime 4 - 8 JAM
# ----------------------------------------------------------
print(f"\n{sep}")
print("  [B] BARIS DENGAN downtime_hours ANTARA 4 DAN 8 JAM (inklusif)")
print(sep)
print()

df_range_4_8 = df_corrective[df_corrective["downtime_hours"].between(4, 8)]

if df_range_4_8.empty:
    print("  Tidak ada data di range 4-8 jam.")
    print("  → Ini konfirmasi Score 2 memang tidak bisa muncul dari data ini.")
else:
    print(f"  Ditemukan {len(df_range_4_8)} baris:")
    print()
    print(df_range_4_8[
        ["machine_id", "date", "maintenance_type",
         "downtime_hours", "technician_notes"]
    ].to_string(index=False))

# ----------------------------------------------------------
# [4] DISTRIBUSI DALAM RANGE BUCKET
# ----------------------------------------------------------
print(f"\n{sep}")
print("  [C] DISTRIBUSI downtime_hours PER BUCKET (Corrective)")
print(sep)
print()

total = len(df_corrective)

n_lt4  = int((dh < 4).sum())
n_4to8 = int(dh.between(4, 8).sum())
n_gt8  = int((dh > 8).sum())

pct_lt4  = n_lt4  / total * 100 if total > 0 else 0
pct_4to8 = n_4to8 / total * 100 if total > 0 else 0
pct_gt8  = n_gt8  / total * 100 if total > 0 else 0

print(f"  {'Bucket':<15} {'Jumlah':>8}  {'Persentase':>12}")
print(f"  {'-'*40}")
print(f"  {'< 4 jam':<15} {n_lt4:>8,}  {pct_lt4:>11.2f}%")
print(f"  {'4 - 8 jam':<15} {n_4to8:>8,}  {pct_4to8:>11.2f}%")
print(f"  {'> 8 jam':<15} {n_gt8:>8,}  {pct_gt8:>11.2f}%")
print(f"  {'-'*40}")
print(f"  {'TOTAL':<15} {total:>8,}  {'100.00%':>12}")

print(f"\n{SEP}")
print("  KESIMPULAN INVESTIGASI (DFT-03)")
print(SEP)
if n_4to8 == 0:
    print("\n  [CONFIRMED] Data downtime bersifat bimodal:")
    print(f"    → Corrective < 4 jam  : {n_lt4:,} baris ({pct_lt4:.1f}%)")
    print(f"    → Corrective > 8 jam  : {n_gt8:,} baris ({pct_gt8:.1f}%)")
    print( "    → Range 4-8 jam       : 0 baris (0.0%) ← gap di data")
    print("\n  Implikasi: severity_score hanya menghasilkan nilai 1 dan 3.")
    print("  Opsi di Fase 8: binarize (0=normal, 1=berat) jika diperlukan.")
else:
    print(f"\n  [INFO] Ditemukan {n_4to8} baris di range 4-8 jam.")
    print("  Score 2 seharusnya muncul — periksa logika assign_severity_score.")
print(SEP)


  INVESTIGASI DISTRIBUSI DOWNTIME — CORRECTIVE ONLY

  Total baris Corrective : 166
  Total baris semua tipe : 500

-----------------------------------------------------------------
  [A] STATISTIK downtime_hours (Corrective)
-----------------------------------------------------------------

  min    : 8.20 jam
  max    : 23.80 jam
  mean   : 15.92 jam
  median : 16.35 jam

  value_counts (top 20):

       8.2 jam  →    2 baris
       8.3 jam  →    2 baris
       8.4 jam  →    2 baris
       8.5 jam  →    2 baris
       8.6 jam  →    3 baris
       8.7 jam  →    2 baris
       9.1 jam  →    2 baris
       9.2 jam  →    1 baris
       9.4 jam  →    3 baris
       9.5 jam  →    1 baris
       9.6 jam  →    2 baris
       9.7 jam  →    2 baris
       9.8 jam  →    1 baris
      10.3 jam  →    1 baris
      10.4 jam  →    3 baris
      10.6 jam  →    2 baris
      10.7 jam  →    2 baris
      10.8 jam  →    2 baris
      11.0 jam  →    3 baris
      11.1 jam  →    2 baris
      11.3 jam  →

4.5 Feature Summary & Export Final Fase 4

### 4. Cross-Sensor Ratios & Degradation Proxy
**Tujuan:** Menghitung 4 rasio cross-sensor (temp/vibration, power/rpm, pressure/temp, noise/vibration) dan 1 fitur degradasi (hours_since_last_maint) dari maintenance logs.  
**Input:** df (100.000×68) dari Cell 3, data/raw/maintenance_logs.csv  
**Output:** df (100.000×73)  +4 kolom ratio + 1 kolom degradation proxy  
**Catatan:** Rasio cross-sensor menangkap interaksi antar sensor yang tidak bisa ditangkap individual. hours_since_last_maint: baris dengan nilai -1 = 2.856 (belum ada maintenance tercatat sebelumnya)  acceptable, bukan bug.  


In [8]:
# ============================================================
# FASE 4 — Cell Terakhir: Feature Inventory Audit & Export
# ============================================================

import sys
import numpy as np
import pandas as pd
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
ML_ROOT      = NOTEBOOK_DIR.parent.parent
SRC_PATH     = ML_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from config import DATA_INTERIM_DIR

SEP = "=" * 65
sep = "-" * 65

# ════════════════════════════════════════════════════════════
# BAGIAN 1 — FEATURE INVENTORY AUDIT
# ════════════════════════════════════════════════════════════

print(SEP)
print("  BAGIAN 1 — FEATURE INVENTORY AUDIT")
print(SEP)

GRUP_ORIGINAL = [
    "timestamp", "machine_id", "temperature", "vibration",
    "pressure", "rpm", "power_consumption", "noise_level",
    "humidity", "operating_hours", "failure",
]
GRUP_LABEL = [
    "health_label", "health_label_confirmed", "health_label_encoded",
]
GRUP_ROLLING     = [c for c in df.columns if "_roll_" in c]
GRUP_LAG         = [c for c in df.columns if "_lag_"  in c]
GRUP_RATIO       = [
    "temp_per_vibration", "power_per_rpm",
    "pressure_per_temp", "noise_per_vibration",
]
GRUP_DEGRADATION = ["hours_since_last_maint"]
GRUP_NLP         = ["damage_category", "severity_score"]

# Hanya sertakan kolom yang benar-benar ada di df (defensive)
feature_groups = {
    "Original"    : [c for c in GRUP_ORIGINAL    if c in df.columns],
    "Label"       : [c for c in GRUP_LABEL        if c in df.columns],
    "Rolling"     : [c for c in GRUP_ROLLING      if c in df.columns],
    "Lag"         : [c for c in GRUP_LAG          if c in df.columns],
    "Ratio"       : [c for c in GRUP_RATIO        if c in df.columns],
    "Degradation" : [c for c in GRUP_DEGRADATION  if c in df.columns],
    "NLP"         : [c for c in GRUP_NLP          if c in df.columns],
}

total_catalogued = sum(len(v) for v in feature_groups.values())
all_df_cols      = set(df.columns)
catalogued_cols  = set(c for v in feature_groups.values() for c in v)
uncatalogued     = all_df_cols - catalogued_cols

print()
print(f"  {'Grup':<16} {'Jumlah':>7}  Kolom")
print(f"  {'-'*62}")
for grup, cols in feature_groups.items():
    col_str = ", ".join(cols) if cols else "(kosong)"
    # Tampilkan baris pertama dengan nama grup
    print(f"  {grup:<16} {len(cols):>7}  {col_str[:45]}")
    # Jika kolom banyak, lanjutkan wrap
    if len(col_str) > 45:
        remaining = col_str[45:]
        chunks = [remaining[i:i+62] for i in range(0, len(remaining), 62)]
        for chunk in chunks:
            print(f"  {'':<25}{chunk}")
print(f"  {'-'*62}")
print(f"  {'TOTAL':<16} {total_catalogued:>7}  kolom terkatalog")
print(f"  Total kolom df    : {len(df.columns)}")

if uncatalogued:
    print(f"\n  [WARN] {len(uncatalogued)} kolom TIDAK terkatalog: {sorted(uncatalogued)}")
else:
    print(f"\n  [OK] Semua {len(df.columns)} kolom berhasil dikatalog.")

# ════════════════════════════════════════════════════════════
# BAGIAN 2 — FINAL QUALITY CHECK
# ════════════════════════════════════════════════════════════

print(f"\n{SEP}")
print("  BAGIAN 2 — FINAL QUALITY CHECK")
print(SEP)

checks = {}

# [1] NaN
total_nan = int(df.isna().sum().sum())
checks["NaN = 0"]              = total_nan == 0
print(f"\n  [1] Total NaN di seluruh df       : {total_nan:,}  "
      f"→  {'PASS' if checks['NaN = 0'] else 'FAIL'}")

# [2] Inf
num_cols  = df.select_dtypes(include=[np.number]).columns
total_inf = int(np.isinf(df[num_cols].values).sum())
checks["Inf = 0"]              = total_inf == 0
print(f"  [2] Total Inf di kolom numerik    : {total_inf:,}  "
      f"→  {'PASS' if checks['Inf = 0'] else 'FAIL'}")

# [3] Shape
shape_ok = df.shape == (100_000, 75)
checks["Shape (100k×75)"]      = shape_ok
print(f"  [3] Shape df                      : {df.shape}  "
      f"→  {'PASS' if shape_ok else 'FAIL (expected 100000×75)'}")

# [4] Dtype check
dtype_checks = {
    "timestamp"           : ("datetime64", pd.api.types.is_datetime64_any_dtype),
    "machine_id"          : ("object",     lambda s: s.dtype == object),
    "health_label_encoded": ("int/float",  pd.api.types.is_numeric_dtype),
    "severity_score"      : ("int/float",  pd.api.types.is_numeric_dtype),
    "damage_category"     : ("object",     lambda s: s.dtype == object),
}

print(f"\n  [4] Dtype kolom kritis:")
all_dtype_ok = True
for col, (expected_label, check_fn) in dtype_checks.items():
    if col in df.columns:
        ok  = check_fn(df[col])
        res = "PASS" if ok else f"FAIL (got {df[col].dtype})"
        print(f"      {col:<30} → expected {expected_label:<12} | {res}")
        if not ok:
            all_dtype_ok = False
    else:
        print(f"      {col:<30} → [WARN] Kolom tidak ditemukan di df")
        all_dtype_ok = False

checks["Dtypes OK"] = all_dtype_ok

# Ringkasan akhir
print(f"\n{sep}")
print("  RINGKASAN QUALITY CHECK")
print(sep)
print()
all_pass = all(checks.values())
for check_name, result in checks.items():
    icon = "[PASS]" if result else "[FAIL]"
    print(f"  {icon}  {check_name}")

print()
if all_pass:
    print("  [OK] SEMUA CHECK LULUS — df siap untuk di-export.")
else:
    print("  [WARN] ADA CHECK YANG GAGAL — periksa sebelum export.")

# ════════════════════════════════════════════════════════════
# BAGIAN 3 — EXPORT (PARQUET + CSV)
# ════════════════════════════════════════════════════════════

print(f"\n{SEP}")
print("  BAGIAN 3 — EXPORT FINAL FASE 4")
print(SEP)

DATA_INTERIM_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_PATH = DATA_INTERIM_DIR / "df_sensor_featured.parquet"
CSV_PATH     = DATA_INTERIM_DIR / "df_sensor_featured_preview.csv"

# --- Export Parquet ---
df.to_parquet(PARQUET_PATH, index=False)
parquet_size_mb = PARQUET_PATH.stat().st_size / (1024 ** 2)

# --- Export CSV ---
df.to_csv(CSV_PATH, index=False)
csv_size_mb = CSV_PATH.stat().st_size / (1024 ** 2)

print(f"\n{sep}")
print("  [FORMAT 1] Parquet — Pipeline Resmi")
print(sep)
print(f"\n  Path   : {PARQUET_PATH}")
print(f"  Shape  : {df.shape}")
print(f"  Ukuran : {parquet_size_mb:.2f} MB")

print(f"\n{sep}")
print("  [FORMAT 2] CSV — Preview Manual")
print(sep)
print(f"\n  Path   : {CSV_PATH}")
print(f"  Shape  : {df.shape}")
print(f"  Ukuran : {csv_size_mb:.2f} MB")

print(f"\n{sep}")
print("  CATATAN")
print(sep)
print("\n  .parquet = pipeline resmi (Fase 5 dst.)")
print("  .csv     = preview manual (Excel) saja")

print(f"\n{SEP}")
print("  [OK] Feature Engineering Fase 4 SELESAI.")
print(f"  [OK] Output : df_sensor_featured.parquet  ({df.shape[0]:,} baris × {df.shape[1]} kolom)")
print(SEP)


  BAGIAN 1 — FEATURE INVENTORY AUDIT

  Grup              Jumlah  Kolom
  --------------------------------------------------------------
  Original              11  timestamp, machine_id, temperature, vibration
                           , pressure, rpm, power_consumption, noise_level, humidity, ope
                           rating_hours, failure
  Label                  3  health_label, health_label_confirmed, health_
                           label_encoded
  Rolling               36  temperature_roll_mean_24h, temperature_roll_s
                           td_24h, temperature_roll_max_24h, temperature_roll_mean_48h, t
                           emperature_roll_std_48h, temperature_roll_max_48h, vibration_r
                           oll_mean_24h, vibration_roll_std_24h, vibration_roll_max_24h, 
                           vibration_roll_mean_48h, vibration_roll_std_48h, vibration_rol
                           l_max_48h, pressure_roll_mean_24h, pressure_roll_std_24h, pres
           